# Release pipeline

A release decision that lives in a notebook is a wish. This notebook turns yours into a pipeline: a gate that exits 1, a manifest a security review can read, a canary verdict that refuses to promote on noise, an authorisation check that runs before retrieval, and a checklist of what only your platform team can answer.

## Learn | Create | Grow

### Learn
A release gate as one line in a pipeline, the eight properties of a manifest a platform team greps for, and why a canary at forty samples cannot tell an improvement from luck.

### Create
The gate, the manifest check, and the canary verdict run on your own eval results and written to the workspace, then an authorisation check over your corpus and a deployment checklist with the unknowns marked.

### Grow
A deployment is a conversation with whoever runs the platform. Bring them the manifest and the checklist, and ask what is wrong with them.

**Estimated time:** 40 minutes
**Reads:** deepeval_results, release_decision, eval_cases, trajectories, tools_catalog, corpus
**Writes:** eval_gate, canary_verdict, deploy_checklist

## Setup

No key, no model, no cluster. Everything reads from the workspace, with the seed as fallback, plus the two files in `deploy/` beside this notebook. The two scripts the pipeline calls live in the repository's `scripts/` folder, and the cells below run them the way a pipeline step would: as a subprocess with an exit code.

In [1]:
import math, re, subprocess, sys
from collections import defaultdict

import pandas as pd
from IPython.display import Markdown, display

from helpers import workspace as ws
from helpers.config import ROOT, LLM_MODEL, LLM_BASE
from helpers.evals import fingerprint, gate
from helpers.paths import local

RESULTS = ws.load("deepeval_results")
DECISION = ws.load("release_decision")
EVAL_CASES = ws.load("eval_cases")
TRAJECTORIES = ws.load("trajectories")
TOOLS = ws.load("tools_catalog")
CORPUS_DIR = ws.load_path("corpus")
PAGES = [{"name": str(p.relative_to(CORPUS_DIR)), "text": p.read_text(encoding="utf-8")}
         for p in sorted(CORPUS_DIR.rglob("*.md"))]

DEPLOY = local("deploy")
SCRIPTS = ROOT / "scripts"
VERSIONS = sorted({r["version"] for r in RESULTS})
print(f"✅ offline; {len(RESULTS)} result rows over versions {VERSIONS}; {len(EVAL_CASES)} eval cases; "
      f"{len(PAGES)} corpus pages; deploy files {sorted(p.name for p in DEPLOY.iterdir())}; "
      f"source: {ws.source('deepeval_results')}")

✅ offline; 30 result rows over versions ['v1', 'v2']; 5 eval cases; 26 corpus pages; deploy files ['deploy.yml', 'k8s.yaml']; source: workspace


You should see a ✅ line with the result row count, two version names, the eval case and corpus page counts, and the two deploy files. Stop here if the versions list has fewer than two entries: the canary needs a baseline and a candidate, so run the eval notebook that writes the results first.

# Learn

## Task 1 of 6 — A pass rate per metric, and the average that hides a failure

Your eval results are one row per test, version, and metric. The first gate most teams write averages everything into one number and compares it with a bar. Compute that, then the pass rate of each metric on its own, and hold the candidate version to a floor per metric with `gate`. The floors in `MINIMUM` are yours to edit. The saved gate records them, the rates, and a fingerprint of the cases it measured, so the result says what was tested and against what.

In [2]:
def pass_rates(rows: list[dict]) -> dict[str, dict[str, float]]:
    """{version: {metric: fraction of rows that passed}}. An errored row counts as a failure."""
    seen, hits = defaultdict(int), defaultdict(int)
    for r in rows:
        key = (r["version"], r["metric"])
        seen[key] += 1
        hits[key] += 1 if r.get("passed") and not r.get("error") else 0
    out = defaultdict(dict)
    for (version, metric), n in seen.items():
        out[version][metric] = round(hits[(version, metric)] / n, 3)
    return dict(out)


RATES = pass_rates(RESULTS)
BASELINE, CANDIDATE = VERSIONS[0], VERSIONS[-1]
BAR = 0.8
for v in VERSIONS:                                       # the weak gate: one average
    mean = sum(RATES[v].values()) / len(RATES[v])
    print(f"{v}: mean pass rate {mean:.2f}, {'clears' if mean >= BAR else 'under'} the bar of {BAR}")
display(pd.DataFrame(RATES).T)                           # the strong one: every metric visible

MINIMUM = {metric: BAR for metric in RATES[CANDIDATE]}   # edit the floors here
VERDICT = gate(RATES[CANDIDATE], MINIMUM)
print(f"{CANDIDATE} per metric: {'passed' if VERDICT['passed'] else 'FAILED on ' + ', '.join(VERDICT['failed'])}")

MEASURED = [{"id": t} for t in sorted({r["test"] for r in RESULTS})]
EVAL_GATE = {"passed": VERDICT["passed"], "failed": VERDICT["failed"], "minimum": MINIMUM, "version": CANDIDATE,
             "rates": RATES[CANDIDATE], "cases_measured": len(MEASURED), "cases_written": len(EVAL_CASES),
             "fingerprint": fingerprint(MEASURED), "fingerprint_all_cases": fingerprint(EVAL_CASES)}
print(f"fingerprint {EVAL_GATE['fingerprint']} over {len(MEASURED)} of {len(EVAL_CASES)} cases")
ws.save("eval_gate", EVAL_GATE)

v1: mean pass rate 0.73, under the bar of 0.8
v2: mean pass rate 0.67, under the bar of 0.8


,answer_relevancy,faithfulness,correctness
v1,1.0,0.8,0.4
v2,0.8,0.8,0.4


v2 per metric: FAILED on correctness
fingerprint 05f8542ce58b over 5 of 5 cases
✅ wrote eval_gate → workspace/demo/eval_gate.json (9 rows)


PosixPath('/Users/praveen.pattanshetti/AIPProvisioning/titanium-engineer-labs-adapt/workspace/demo/eval_gate.json')

You should see the mean per version, a table with one row per version and one column per metric, a per-metric verdict for the candidate, a fingerprint line, and a ✅ line. Stop here if the table has one column: the results carry a single metric, and a gate over one number is the weak version again.

## Task 2 of 6 — The same gate as a script, and the line that runs it

A gate is only a gate when a pipeline can call it and read the answer. `scripts/release_gate.py` does what the cell above did and turns the verdict into an exit code: 0 ships, 1 stops the job. Run it twice as a subprocess, with a floor the candidate clears and with the floor you meant, and read the exit codes. Then find the one line in `deploy/deploy.yml` that calls it. Everything else in that file is a build and a deploy any app would need.

In [3]:
def run_script(name: str, *args: str, stdin: str | None = None) -> int:
    """Run one of the repository's scripts the way a pipeline step would, and print what it said."""
    proc = subprocess.run([sys.executable, str(SCRIPTS / name), *args], input=stdin,
                          capture_output=True, text=True, cwd=ROOT)
    print((proc.stdout + proc.stderr).strip())
    print(f"exit code {proc.returncode}\n")
    return proc.returncode


run_script("release_gate.py", "--min-score", "0.5")
run_script("release_gate.py", "--min-score", str(BAR))

WORKFLOW = (DEPLOY / "deploy.yml").read_text(encoding="utf-8")
for n, line in enumerate(WORKFLOW.splitlines(), 1):
    if "release_gate" in line:
        print(f"deploy.yml line {n}: {line.strip()}")
print("steps:", " -> ".join(re.findall(r"- name: (\w[\w ]*)", WORKFLOW)))

/Users/praveen.pattanshetti/AIPProvisioning/titanium-engineer-labs-adapt/.venv/bin/python3: can't open file '/Users/praveen.pattanshetti/AIPProvisioning/titanium-engineer-labs-adapt/scripts/release_gate.py': [Errno 2] No such file or directory
exit code 2

/Users/praveen.pattanshetti/AIPProvisioning/titanium-engineer-labs-adapt/.venv/bin/python3: can't open file '/Users/praveen.pattanshetti/AIPProvisioning/titanium-engineer-labs-adapt/scripts/release_gate.py': [Errno 2] No such file or directory
exit code 2

deploy.yml line 31: uv run python scripts/release_gate.py --min-score 0.8
steps: Test -> Gate                      -> Build and push -> Deploy


You should see the rate table twice, exit code 0 on the low floor and 1 on the real one, then the workflow line that calls the gate and the four step names. Stop here if both exit codes are 0: the candidate clears every floor, so rerun with a floor of 0.99 and confirm the gate can refuse.

### ❓ Question
The gate reads results a judge wrote earlier. What has to be true about that judge run for an exit code of 0 to mean the release is safe, and which of those things does the script check?

Answer:

## Task 3 of 6 — A manifest a security review can read

`deploy/k8s.yaml` describes the demo app as a Deployment, a Service, and a PodDisruptionBudget. Eight properties in it are what a platform team greps for before reading anything else: non-root, a read-only filesystem, every capability dropped, no privilege escalation, an image pinned to a tag, requests and limits, two probes, and a disruption budget once there is more than one replica. `scripts/check_manifests.py` asserts all eight with no YAML library. Run it on the file, then break one property in memory and run it again on standard input.

In [4]:
MANIFEST = (DEPLOY / "k8s.yaml").read_text(encoding="utf-8")
NEEDLES = ["runAsNonRoot: true", "readOnlyRootFilesystem: true", "- ALL", "allowPrivilegeEscalation: false",
           "demo-app:0.1.0", "requests:", "limits:", "readinessProbe:", "livenessProbe:", "kind: PodDisruptionBudget"]
for needle in NEEDLES:
    print(f"{'✅' if needle in MANIFEST else 'missing'} {needle}")
print()

run_script("check_manifests.py", "-", stdin=MANIFEST)

BROKEN = MANIFEST.replace("runAsNonRoot: true", "runAsNonRoot: false")   # one line, the most common finding
assert BROKEN != MANIFEST
run_script("check_manifests.py", "-", stdin=BROKEN)

✅ runAsNonRoot: true
✅ readOnlyRootFilesystem: true
✅ - ALL
✅ allowPrivilegeEscalation: false
✅ demo-app:0.1.0
✅ requests:
✅ limits:
✅ readinessProbe:
✅ livenessProbe:
✅ kind: PodDisruptionBudget

/Users/praveen.pattanshetti/AIPProvisioning/titanium-engineer-labs-adapt/.venv/bin/python3: can't open file '/Users/praveen.pattanshetti/AIPProvisioning/titanium-engineer-labs-adapt/scripts/check_manifests.py': [Errno 2] No such file or directory
exit code 2



/Users/praveen.pattanshetti/AIPProvisioning/titanium-engineer-labs-adapt/.venv/bin/python3: can't open file '/Users/praveen.pattanshetti/AIPProvisioning/titanium-engineer-labs-adapt/scripts/check_manifests.py': [Errno 2] No such file or directory
exit code 2



2

You should see ten ✅ lines, a pass line from the check with exit code 0, then one failure line naming the pod that runs as root with exit code 1. Stop here if the first run already fails: the manifest beside this notebook was edited, so read the failure line and fix the property it names.

## Task 4 of 6 — Rolling, canary, shadow, and a verdict that refuses to promote on noise

The workflow does a rolling update, which answers one question: does the new version start and stay up. A worse model passes that. A canary sends a slice of real traffic to the candidate; a shadow copies traffic to it and discards the replies. Both produce a pass rate on a sample, and a sample of forty moves around. Write the verdict from scratch: latency is a hard gate, too few samples is a hold, and after that the difference of two proportions with its 95% interval decides. Then apply it to your own two versions.

In [5]:
def naive_verdict(baseline_pass: float, candidate_pass: float) -> str:
    return "promote" if candidate_pass > baseline_pass else "hold"


def canary_verdict(baseline_pass: float, baseline_n: int, candidate_pass: float, candidate_n: int, *,
                   min_samples: int, p95_ms: float | None, p95_limit_ms: float) -> dict:
    """Promote, hold, or rollback. Hard gates first, statistics last."""
    out = {"delta": None, "low": None, "high": None}
    if baseline_n and candidate_n:
        se = math.sqrt(baseline_pass * (1 - baseline_pass) / baseline_n
                       + candidate_pass * (1 - candidate_pass) / candidate_n)
        delta = candidate_pass - baseline_pass
        out = {"delta": round(delta, 3), "low": round(delta - 1.96 * se, 3), "high": round(delta + 1.96 * se, 3)}
    if p95_ms is not None and p95_ms > p95_limit_ms:
        return {"verdict": "rollback", "why": f"p95 {p95_ms:.0f} ms is over the {p95_limit_ms:.0f} ms limit", **out}
    holds = []
    if p95_ms is None:
        holds.append("p95 latency not measured")
    if candidate_n < min_samples:
        holds.append(f"{candidate_n} samples, need {min_samples}")
    if holds:
        return {"verdict": "hold", "why": "; ".join(holds), **out}
    interval = f"{out['delta']:+.3f} [{out['low']:+.3f}, {out['high']:+.3f}]"
    if out["high"] < 0:
        return {"verdict": "rollback", "why": f"worse: {interval}", **out}
    if out["low"] > 0:
        return {"verdict": "promote", "why": f"better: {interval}", **out}
    return {"verdict": "hold", "why": f"interval spans zero: {interval}", **out}


BASE_RATE, BASE_N, MIN_SAMPLES, P95_LIMIT_MS = 0.81, 4000, 200, 1200
table = []
for rate in (0.95, 0.86, 0.83, 0.72):
    for n in (40, 400, 4000):
        v = canary_verdict(BASE_RATE, BASE_N, rate, n, min_samples=MIN_SAMPLES, p95_ms=900, p95_limit_ms=P95_LIMIT_MS)
        table.append({"candidate": rate, "n": n, "p95_ms": 900, "naive": naive_verdict(BASE_RATE, rate),
                      "verdict": v["verdict"], "why": v["why"]})
v = canary_verdict(BASE_RATE, BASE_N, 0.93, 4000, min_samples=MIN_SAMPLES, p95_ms=1500, p95_limit_ms=P95_LIMIT_MS)
table.append({"candidate": 0.93, "n": 4000, "p95_ms": 1500, "naive": naive_verdict(BASE_RATE, 0.93),
              "verdict": v["verdict"], "why": v["why"]})
display(pd.DataFrame(table))


def case_pass(rows: list[dict], version: str) -> dict[str, bool]:
    """A case passes a version only when every metric passed."""
    flags = defaultdict(list)
    for r in rows:
        if r["version"] == version:
            flags[r["test"]].append(bool(r.get("passed")) and not r.get("error"))
    return {case: all(f) for case, f in flags.items()}


base, cand = case_pass(RESULTS, BASELINE), case_pass(RESULTS, CANDIDATE)
base_rate, cand_rate = sum(base.values()) / len(base), sum(cand.values()) / len(cand)
P95_MS = None          # the eval results carry no latency; a load test fills this in
OWN = canary_verdict(base_rate, len(base), cand_rate, len(cand),
                     min_samples=MIN_SAMPLES, p95_ms=P95_MS, p95_limit_ms=P95_LIMIT_MS)
CANARY = {**OWN, "baseline": {"version": BASELINE, "pass_rate": round(base_rate, 3), "n": len(base)},
          "candidate": {"version": CANDIDATE, "pass_rate": round(cand_rate, 3), "n": len(cand)},
          "min_samples": MIN_SAMPLES, "p95_ms": P95_MS, "p95_limit_ms": P95_LIMIT_MS}
print(f"{BASELINE} {base_rate:.2f} (n={len(base)}) -> {CANDIDATE} {cand_rate:.2f} (n={len(cand)}): "
      f"{OWN['verdict']}, {OWN['why']}")
ws.save("canary_verdict", CANARY)

,candidate,n,p95_ms,naive,verdict,why
0,0.95,40,900,promote,hold,"40 samples, need 200"
1,0.95,400,900,promote,promote,"better: +0.140 [+0.115, +0.165]"
2,0.95,4000,900,promote,promote,"better: +0.140 [+0.126, +0.154]"
3,0.86,40,900,promote,hold,"40 samples, need 200"
4,0.86,400,900,promote,promote,"better: +0.050 [+0.014, +0.086]"
5,0.86,4000,900,promote,promote,"better: +0.050 [+0.034, +0.066]"
6,0.83,40,900,promote,hold,"40 samples, need 200"
7,0.83,400,900,promote,hold,"interval spans zero: +0.020 [-0.019, +0.059]"
8,0.83,4000,900,promote,promote,"better: +0.020 [+0.003, +0.037]"
9,0.72,40,900,hold,hold,"40 samples, need 200"


v1 0.40 (n=5) -> v2 0.20 (n=5): hold, p95 latency not measured; 5 samples, need 200
✅ wrote canary_verdict → workspace/demo/canary_verdict.json (10 rows)


PosixPath('/Users/praveen.pattanshetti/AIPProvisioning/titanium-engineer-labs-adapt/workspace/demo/canary_verdict.json')

You should see a thirteen-row table where naive says promote on every higher rate while verdict holds at n = 40, promotes only when the interval clears zero, and rolls back the slow row, then your own verdict and a ✅ line. Stop here if your own verdict is promote: a handful of cases cannot earn that, so check `MIN_SAMPLES`.

### ❓ Question
Your own verdict holds on sample size. At your product's real request volume, how long would a canary at five percent of traffic need to run before the interval on your pass rate could clear zero, and does that make shadowing offline the only honest option?

Answer:

# Create

## Task 5 of 6 — Who may read what, before retrieval

Your app will sit behind a gateway that authenticates the user and forwards an identity in a header. Authentication says who; authorisation says what they may read, and that check is an if statement in code, never a sentence in a prompt. The trap is retrieval: an index holding pages with different access rules is an authorisation bypass. Tag your corpus pages with groups, then run the same search twice for a newcomer who may read the knowledge base only: filtering after ranking, then filtering before. Watch what the first one gives away.

In [6]:
def identity_from_headers(headers: dict) -> dict | None:
    """Trust the gateway in front of the app: read the identity it forwarded."""
    lower = {k.lower(): v for k, v in headers.items()}
    user = lower.get("x-forwarded-user", "").strip()
    if not user:
        return None
    groups = [g.strip() for g in lower.get("x-forwarded-groups", "").split(",") if g.strip()]
    return {"user": user, "groups": groups}


def may_access(identity: dict | None, document_groups: list[str]) -> bool:
    """Authorisation: an if statement, not a prompt."""
    return bool(identity) and bool(set(identity["groups"]) & set(document_groups))


GROUPS = {"kb": ["all-staff"], "wiki": ["all-staff"], "charter.md": ["all-staff"],
          "transcripts": ["helpdesk-agents"], "prompts": ["engineering"], "vibe_checks.md": ["engineering"]}


def groups_of(page: dict) -> list[str]:
    return GROUPS.get(page["name"].split("/")[0], ["engineering"])


STOP = {"a", "an", "and", "are", "as", "at", "be", "but", "for", "from", "how", "i", "in", "is", "it",
        "of", "on", "or", "the", "to", "what", "when", "with", "my", "can", "do", "you", "me"}


def terms(text: str) -> set[str]:
    return {t for t in re.findall(r"[a-z0-9][a-z0-9-]+", text.lower()) if t not in STOP}


def rank(pages: list[dict], query: str, k: int = 3) -> list[dict]:
    q = terms(query)
    ranked = sorted(pages, key=lambda p: len(q & terms(p["text"])), reverse=True)
    return [p for p in ranked[:k] if q & terms(p["text"])]


def search_then_filter(query: str, identity: dict | None) -> tuple[list[str], str]:   # the trap
    hits = rank(PAGES, query)
    allowed = [p["name"] for p in hits if may_access(identity, groups_of(p))]
    hidden = [p["name"] for p in hits if p["name"] not in allowed]
    return allowed, f"{len(hits)} matched, {len(hidden)} hidden: {hidden}"


def filter_then_search(query: str, identity: dict | None) -> tuple[list[str], str]:   # the fix
    readable = [p for p in PAGES if may_access(identity, groups_of(p))]
    hits = rank(readable, query)
    return [p["name"] for p in hits], f"{len(hits)} matched"


newcomer = identity_from_headers({"X-Forwarded-User": "new.starter@example.com", "X-Forwarded-Groups": "all-staff"})
QUERY = EVAL_CASES[0]["question"]
print("query:", QUERY)
for fn in (search_then_filter, filter_then_search):
    names, note = fn(QUERY, newcomer)
    print(f"{fn.__name__:<19} {names}  ({note})")
print("no header at all:", identity_from_headers({}), "->", filter_then_search(QUERY, None)[0])

query: My VPN connects but I cannot reach staging.
search_then_filter  ['charter.md']  (3 matched, 2 hidden: ['prompts/meta-prompt-applied.md', 'prompts/meta-prompt-generate.md'])
filter_then_search  ['charter.md', 'wiki/index.md']  (2 matched)
no header at all: None -> []


You should see the query, the trap line listing readable pages plus a note naming the hidden ones, the fix line with no such note, and an empty list for a request with no header. Stop here if the trap hides nothing: pick a query from a transcript so the ranking reaches a page the newcomer may not read.

⚠️ Trusting that header is safe only when nothing can reach the app except through the gateway. A service that is addressable inside the network lets anyone set `X-Forwarded-User` to whoever they like. The checklist below asks that question, and the answer has to be no.

## Task 6 of 6 — The deployment checklist

Thirty questions decide whether this app can be deployed where you work, and most are answered by a person, not a file. Some the repository answers already: the health endpoint is the probe path in the manifest, the secret comes from `.env` today and a Secret on the cluster, the app keeps its conversation in process memory. The cell derives those answers from the files themselves, marks the rest unknown, and saves the list as markdown. Unknown is an answer someone can act on; a blank is not.

In [7]:
APP_SRC = (ROOT / "project" / "app" / "app.py").read_text(encoding="utf-8")
CONFIG_SRC = (ROOT / "helpers" / "config.py").read_text(encoding="utf-8")
probe = re.search(r"path:\s*(\S+)", MANIFEST).group(1)
secret = re.search(r"secretRef:\s*\n\s*name:\s*(\S+)", MANIFEST).group(1)
image = re.search(r"image:\s*(\S+)", MANIFEST).group(1)
asks = re.search(r"requests:.*?\n\s*cpu:\s*(\S+)\n\s*memory:\s*(\S+)", MANIFEST, re.S)
outward = [f"{t['name']} ({t['kind']})" for t in TOOLS["tools"] if t.get("kind") in ("mcp", "utcp", "sub-agent")]
tool_steps = sum(1 for t in TRAJECTORIES for s in t["steps"] if s.get("role") == "tool")
decision = next((ln.lstrip("# ").strip() for ln in DECISION.splitlines() if ln.startswith("# ")), "no decision saved")
endpoint = "a self-hosted endpoint (OPENAI_BASE_URL is set)" if LLM_BASE else "the provider's default endpoint"
stateful = "session_state" in APP_SRC
dotenv = "load_dotenv" in CONFIG_SRC

CHECKLIST = [   # (area, question, status, answer, source)
    ("Compute", "Where do internal apps run, and who owns that cluster?", "unknown", "", ""),
    ("Compute", "What does the app need: CPU, memory, a GPU?", "answered",
     f"requests {asks.group(1)} CPU and {asks.group(2)} memory; no GPU, the model runs behind {endpoint}", "deploy/k8s.yaml"),
    ("Compute", "Is the app stateless?", "partly" if stateful else "answered",
     "the conversation lives in st.session_state, in the process; two replicas need sticky sessions or a store"
     if stateful else "no per-process state found", "project/app/app.py"),
    ("Compute", "What is the health endpoint?", "answered", f"{probe}, Streamlit's own route, used by both probes", "deploy/k8s.yaml"),
    ("Images", "Is there an internal registry, and what is the image called?", "partly",
     f"{image} is a placeholder until the registry has a name", "deploy/k8s.yaml"),
    ("Images", "Is there a base image you must start from?", "unknown", "", ""),
    ("Images", "Must an image be scanned or signed before it may run?", "unknown", "", ""),
    ("Identity", "How do users authenticate?", "partly",
     "the app has no login of its own; a gateway must terminate SSO in front of it and forward the identity", "project/app/app.py"),
    ("Identity", "Is the service reachable except through the gateway?", "unknown", "", ""),
    ("Identity", "How does the app authenticate outward, to the model?", "answered" if dotenv else "partly",
     "an API key read from .env by helpers.config, taken from a Secret on the cluster", "helpers/config.py"),
    ("Identity", "Is workload identity available, or is it secrets?", "unknown", "", ""),
    ("Secrets", "Where do secrets live today?", "answered", ".env at the repository root, loaded by helpers.config", "helpers/config.py"),
    ("Secrets", "Where will they live on the cluster?", "answered", f"the Secret {secret}, injected with envFrom, never in the image", "deploy/k8s.yaml"),
    ("Secrets", "How are they rotated, and who can read them?", "unknown", "", ""),
    ("Network", "Can the app reach the internet, and through a proxy?", "unknown", "", ""),
    ("Network", "What does the app call?", "partly",
     f"the model at {endpoint}; the tools catalog names {len(outward)} outward capabilities: {', '.join(outward) or 'none'}", "tools_catalog"),
    ("Network", "What is on the egress allowlist, and who adds to it?", "unknown", "", ""),
    ("Network", "Which network is it in, and what can it reach internally?", "unknown", "", ""),
    ("Data", "What classification may this app process?", "unknown", "", ""),
    ("Data", "May data leave the region?", "unknown", "", ""),
    ("Data", "How long may prompts and responses be kept?", "partly",
     f"the workspace already keeps {len(TRAJECTORIES)} full trajectories with no retention rule", "trajectories"),
    ("Data", "Is there a DLP scan on egress?", "unknown", "", ""),
    ("Models", "Is there an approved internal model endpoint?", "partly",
     f"today {LLM_MODEL} at {endpoint}; whether it is approved is a question for the platform team", ".env"),
    ("Models", "May you download open weights, and from where?", "unknown", "", ""),
    ("Models", "What licence does the model carry, and who signs off on it?", "unknown", "", ""),
    ("Legacy", "What must this integrate with, and who owns it?", "partly", TOOLS.get("capability", "see the tools catalog"), "tools_catalog"),
    ("Legacy", "Is it real-time or batch?", "answered",
     f"real-time: {tool_steps} tool calls made at request time across {len(TRAJECTORIES)} trajectories", "trajectories"),
    ("Process", "Who approves a deployment?", "partly", f"the saved decision says '{decision}'; nobody is named", "release_decision"),
    ("Process", "What review gates a deployment?", "answered",
     "release_gate.py and check_manifests.py in the Gate step; both exit 1 to stop the job", "deploy/deploy.yml"),
    ("Process", "What is the rollback, and who is on call?", "partly",
     "kubectl rollout undo to the previous tag in the Deploy step; on call is unknown", "deploy/deploy.yml"),
]
assert len(CHECKLIST) == 30


def render_checklist(items: list[tuple]) -> str:
    counts = defaultdict(int)
    for _, _, status, _, _ in items:
        counts[status] += 1
    lines = ["# Deployment checklist", "",
             f"{len(items)} questions: {counts['answered']} answered by the repository, {counts['partly']} partly, "
             f"{counts['unknown']} unknown. An unknown stays on the list until a person answers it.", ""]
    for area in dict.fromkeys(a for a, *_ in items):
        lines += [f"## {area}", "", "| # | question | status | answer | source |", "|---|---|---|---|---|"]
        for n, (a, q, status, answer, source) in enumerate(items, 1):
            if a == area:
                lines.append(f"| {n} | {q} | {status} | {answer.replace('|', '/')} | {source} |")
        lines.append("")
    return "\n".join(lines)


CHECKLIST_MD = render_checklist(CHECKLIST)
display(Markdown(CHECKLIST_MD))
ws.save("deploy_checklist", CHECKLIST_MD)

# Deployment checklist

30 questions: 7 answered by the repository, 9 partly, 14 unknown. An unknown stays on the list until a person answers it.

## Compute

| # | question | status | answer | source |
|---|---|---|---|---|
| 1 | Where do internal apps run, and who owns that cluster? | unknown |  |  |
| 2 | What does the app need: CPU, memory, a GPU? | answered | requests 250m CPU and 512Mi memory; no GPU, the model runs behind a self-hosted endpoint (OPENAI_BASE_URL is set) | deploy/k8s.yaml |
| 3 | Is the app stateless? | partly | the conversation lives in st.session_state, in the process; two replicas need sticky sessions or a store | project/app/app.py |
| 4 | What is the health endpoint? | answered | /_stcore/health, Streamlit's own route, used by both probes | deploy/k8s.yaml |

## Images

| # | question | status | answer | source |
|---|---|---|---|---|
| 5 | Is there an internal registry, and what is the image called? | partly | registry.example.internal/demo-app:0.1.0 is a placeholder until the registry has a name | deploy/k8s.yaml |
| 6 | Is there a base image you must start from? | unknown |  |  |
| 7 | Must an image be scanned or signed before it may run? | unknown |  |  |

## Identity

| # | question | status | answer | source |
|---|---|---|---|---|
| 8 | How do users authenticate? | partly | the app has no login of its own; a gateway must terminate SSO in front of it and forward the identity | project/app/app.py |
| 9 | Is the service reachable except through the gateway? | unknown |  |  |
| 10 | How does the app authenticate outward, to the model? | answered | an API key read from .env by helpers.config, taken from a Secret on the cluster | helpers/config.py |
| 11 | Is workload identity available, or is it secrets? | unknown |  |  |

## Secrets

| # | question | status | answer | source |
|---|---|---|---|---|
| 12 | Where do secrets live today? | answered | .env at the repository root, loaded by helpers.config | helpers/config.py |
| 13 | Where will they live on the cluster? | answered | the Secret demo-app-secrets, injected with envFrom, never in the image | deploy/k8s.yaml |
| 14 | How are they rotated, and who can read them? | unknown |  |  |

## Network

| # | question | status | answer | source |
|---|---|---|---|---|
| 15 | Can the app reach the internet, and through a proxy? | unknown |  |  |
| 16 | What does the app call? | partly | the model at a self-hosted endpoint (OPENAI_BASE_URL is set); the tools catalog names 3 outward capabilities: mcp_server.py (mcp), ask_trajectory_analyst (sub-agent), eval-lookup-api (utcp) | tools_catalog |
| 17 | What is on the egress allowlist, and who adds to it? | unknown |  |  |
| 18 | Which network is it in, and what can it reach internally? | unknown |  |  |

## Data

| # | question | status | answer | source |
|---|---|---|---|---|
| 19 | What classification may this app process? | unknown |  |  |
| 20 | May data leave the region? | unknown |  |  |
| 21 | How long may prompts and responses be kept? | partly | the workspace already keeps 28 full trajectories with no retention rule | trajectories |
| 22 | Is there a DLP scan on egress? | unknown |  |  |

## Models

| # | question | status | answer | source |
|---|---|---|---|---|
| 23 | Is there an approved internal model endpoint? | partly | today gpt-5.5 at a self-hosted endpoint (OPENAI_BASE_URL is set); whether it is approved is a question for the platform team | .env |
| 24 | May you download open weights, and from where? | unknown |  |  |
| 25 | What licence does the model carry, and who signs off on it? | unknown |  |  |

## Legacy

| # | question | status | answer | source |
|---|---|---|---|---|
| 26 | What must this integrate with, and who owns it? | partly | Look up an eval case or a trajectory by id or by question | tools_catalog |
| 27 | Is it real-time or batch? | answered | real-time: 56 tool calls made at request time across 28 trajectories | trajectories |

## Process

| # | question | status | answer | source |
|---|---|---|---|---|
| 28 | Who approves a deployment? | partly | the saved decision says 'Release decision: HOLD'; nobody is named | release_decision |
| 29 | What review gates a deployment? | answered | release_gate.py and check_manifests.py in the Gate step; both exit 1 to stop the job | deploy/deploy.yml |
| 30 | What is the rollback, and who is on call? | partly | kubectl rollout undo to the previous tag in the Deploy step; on call is unknown | deploy/deploy.yml |


✅ wrote deploy_checklist → workspace/demo/deploy_checklist.md (78 lines)


PosixPath('/Users/praveen.pattanshetti/AIPProvisioning/titanium-engineer-labs-adapt/workspace/demo/deploy_checklist.md')

You should see the rendered checklist in nine areas with a status per row, a first line counting how many are answered, partly, and unknown, then a ✅ line. Stop here if every row says unknown: the manifest or the app source did not load, so check the paths the setup cell printed.

### ❓ Question
Which unknown row would stop the deployment first where you work, and which answered row would your platform team push back on?

Answer:

## Your turn

Answer three of the unknown questions you can find out today, by asking rather than guessing, and name the one person who can answer the rest. Put the answers in `YOUR_ANSWERS` by question number and that person's role in `WHO_KNOWS`, rerun the cell, and explain to a teammate which answer changes the manifest.

In [8]:
# Shape: YOUR_ANSWERS[question number] = what you found out and who told you; WHO_KNOWS = the role who can answer the rest.
YOUR_ANSWERS: dict[int, str] = {}
WHO_KNOWS = ""

UPDATED = [(a, q, "answered" if n in YOUR_ANSWERS else status, YOUR_ANSWERS.get(n, answer),
            "asked" if n in YOUR_ANSWERS else source)
           for n, (a, q, status, answer, source) in enumerate(CHECKLIST, 1)]
if len(YOUR_ANSWERS) >= 3 and WHO_KNOWS:
    CHECKLIST_MD = render_checklist(UPDATED) + f"\nThe rest: ask {WHO_KNOWS}.\n"
    ws.save("deploy_checklist", CHECKLIST_MD)
else:
    print("fill YOUR_ANSWERS with three question numbers and WHO_KNOWS with a role, then rerun")

fill YOUR_ANSWERS with three question numbers and WHO_KNOWS with a role, then rerun


# Grow

## From prototype to production

| What we built | Production equivalent |
|---|---|
| A gate script over one results file | The eval run and the gate as pipeline steps that block the merge |
| Two YAML objects checked by eight rules | A chart or overlay per environment, checked by a policy engine |
| A four-step workflow that deploys on a tag | Image scanning and signing, then dev, staging, and prod with promotion between them |
| A canary verdict in one function | A rollout controller polling live metrics and aborting on its own |
| Group membership from a header | A gateway doing OIDC, a policy engine with roles and attributes, an audit log |
| A checklist in markdown with unknowns | An intake with an owner and a date per question, and a signed-off design document |


## Responsible controls

- The gate's floors agreed before the results exist, and changed only with a recorded reason.
- Authorisation checked before retrieval, on every request, in code that has a test.
- An unknown on the checklist stays visible until a named person answers it.

## Grow further

- Terraform, or your cloud's equivalent, for the two primitives you actually need: a registry to put the image in and a secret store to take the key from. Write both, run the plan, and bring the plan to whoever owns the account.
- Model licences as infrastructure. For the model named in your `.env`, find the licence link on its model card, then the licence file in the repository, then the tag, and answer four questions: commercial use, use restrictions, thresholds, and what a fine-tune inherits. Add the answer to the checklist row that asks.